### Analyse LLM Results 

In [2]:
import os, re, json, time
import pandas as pd
import PyPDF2
from pathlib import Path


In [4]:
# Paths
BASE_DIR    = r'C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026'
PDF_FOLDER  = os.path.join(BASE_DIR, '3-Webscrapping & PDF disclosure', 'pdfs_2026','pdfs')
ANALYSIS_DIR= os.path.join(BASE_DIR, '5-Analysis2026')
RESULTS_DIR = os.path.join(ANALYSIS_DIR, 'llm_results')
PDF_LIST    = os.path.join(ANALYSIS_DIR, 'List_of_all_PDF_for_normal_companies.csv')
INSTRUCTION = os.path.join(BASE_DIR, '4-Github2026_LLM', 'LLM_climaterisk_2026', 'scripts', '3-llm_Instruction.md')
MODEL       = 'claude-sonnet-4-6'
MAX_TOKENS  = 8192
os.makedirs(RESULTS_DIR, exist_ok=True)
print('PDF_FOLDER :', PDF_FOLDER)
print('RESULTS_DIR:', RESULTS_DIR)

PDF_FOLDER : C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure\pdfs_2026\pdfs
RESULTS_DIR: C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\5-Analysis2026\llm_results


In [ ]:
# Load submitted PDFs
df = pd.read_csv(PDF_LIST)
df = df[df['Status'] == 'Submitted'].copy()
print(f'Submitted PDFs: {len(df)}')
df[['Company Name', 'period_year', 'pdf_files']].head(5)

In [ ]:
# Group: one API call per company-year, concatenating all PDFs for that period
groups = df.groupby(['Company Name', 'period_year'])['pdf_files'].apply(list).reset_index()
print(f'Company-year groups to process: {len(groups)}')
groups.head(5)

In [ ]:
# Compile all JSON results into a flat DataFrame
records = []

for json_file in sorted(Path(RESULTS_DIR).glob('*.json')):
    stem  = json_file.stem
    parts = stem.rsplit('_', 1)
    company_slug = parts[0]
    year         = int(parts[1]) if len(parts) == 2 and parts[1].isdigit() else None

    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            questions = json.load(f)
    except Exception as e:
        print(f'  Parse error {json_file.name}: {e}')
        continue

    row = {'file': json_file.name, 'company_slug': company_slug, 'year': year}
    for q in questions:
        qid = q.get('q_id')
        row[f'Q{qid}_answer']     = str(q.get('answer', ''))
        row[f'Q{qid}_confidence'] = q.get('confidence', '')
        row[f'Q{qid}_evidence']   = q.get('evidence', '')
    records.append(row)

llm_results = pd.DataFrame(records)
llm_results.to_csv(os.path.join(ANALYSIS_DIR, 'llm_results_compiled.csv'), index=False)
print(f'Compiled {len(llm_results)} results')
llm_results.head(3)

In [ ]:
# Coverage and confidence summary
total = len(groups)
done  = len(list(Path(RESULTS_DIR).glob('*.json')))
print(f'Processed : {done} / {total}')
print(f'Remaining : {total - done}')

conf_cols    = [c for c in llm_results.columns if c.endswith('_confidence')]
conf_summary = llm_results[conf_cols].apply(pd.Series.value_counts).fillna(0).astype(int)
conf_summary